# Hewwo Welcome to Fake or Real Review Project

# Step 1: Look at the big picture

# Step 2: Get the Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedShuffleSplit, StratifiedKFold, GridSearchCV
from sklearn import svm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
import joblib
import random

In [3]:
reviews_df = pd.read_csv('fake reviews dataset.csv')

In [ ]:
reviews_df.head()

In [ ]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15155 entries, 0 to 15154
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  15155 non-null  object 
 1   rating    15155 non-null  float64
 2   label     15155 non-null  object 
 3   text_     15155 non-null  object 
dtypes: float64(1), object(3)
memory usage: 473.7+ KB


In [4]:
reviews_df.value_counts('label')

,count
label,
CG,20216
OR,20216


In [5]:
reviews_df.describe(include="all")


,category,rating,label,text_
count,40432,40432.000000,40432,40432
unique,10,NaN,2,40412
top,Kindle_Store_5,NaN,CG,My dog loves these and it has kept her occupie...
freq,4730,NaN,20216,2
mean,NaN,4.256579,NaN,NaN
std,NaN,1.144354,NaN,NaN
min,NaN,1.000000,NaN,NaN
25%,NaN,4.000000,NaN,NaN
50%,NaN,5.000000,NaN,NaN
75%,NaN,5.000000,NaN,NaN


In [6]:
reviews_df.value_counts("rating")

,count
rating,
5.0,24559
4.0,7965
3.0,3786
1.0,2155
2.0,1967


# Step 3: Explore the Data (Exploratory Data Analysis - EDA)

In [ ]:
## EDA — Label Distribution & Rating Distribution

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Label distribution pie chart ──
label_counts = reviews_df['label'].value_counts()
label_names  = ['Fake (CG)', 'Real (OR)']
colors = ['#E05252', '#2ecc71']
wedges, texts, autotexts = axes[0].pie(
    label_counts, labels=label_names, autopct='%1.1f%%',
    colors=colors, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for t in autotexts:
    t.set_fontsize(13); t.set_fontweight('bold')
axes[0].set_title('Label Distribution\n(Fake vs. Real Reviews)', fontsize=14, fontweight='bold', pad=15)

# ── Rating distribution bar chart ──
rating_counts = reviews_df['rating'].value_counts().sort_index()
bars = axes[1].bar(
    rating_counts.index.astype(str), rating_counts.values,
    color=['#E05252', '#e67e22', '#f1c40f', '#3498db', '#2ecc71'],
    edgecolor='white', linewidth=1.2
)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{bar.get_height():,}', ha='center', va='bottom',
                 fontsize=10, fontweight='bold', color='#2c3e50')
axes[1].set_xlabel('Star Rating', fontsize=12)
axes[1].set_ylabel('Number of Reviews', fontsize=12)
axes[1].set_title('Rating Distribution\n(1–5 Stars)', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, rating_counts.max() * 1.15)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
## EDA — Reviews per Product Category

fig, ax = plt.subplots(figsize=(12, 6))
cat_counts = reviews_df['category'].value_counts().sort_values()
cat_labels = [c.replace('_5', '').replace('_', ' ') for c in cat_counts.index]

bars = ax.barh(cat_labels, cat_counts.values, color='#0096B7', edgecolor='white', linewidth=0.8)
for bar in bars:
    ax.text(bar.get_width() + 80, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', ha='left', fontsize=10, color='#2c3e50')

ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_title('Review Count by Product Category', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, cat_counts.max() * 1.15)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.show()

## EDA — Review Text Length (Fake vs. Real)

reviews_df['review_length'] = reviews_df['text_'].apply(lambda x: len(str(x).split()))
fake_lengths = reviews_df[reviews_df['label'] == 0]['review_length']
real_lengths = reviews_df[reviews_df['label'] == 1]['review_length']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(fake_lengths, bins=60, alpha=0.65, color='#E05252', label='Fake (CG)', range=(0, 500))
axes[0].hist(real_lengths, bins=60, alpha=0.65, color='#2ecc71', label='Real (OR)', range=(0, 500))
axes[0].set_xlabel('Review Length (words)', fontsize=12)
axes[0].set_ylabel('Number of Reviews', fontsize=12)
axes[0].set_title('Review Length Distribution\nFake vs. Real', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].spines[['top', 'right']].set_visible(False)

data_to_plot = [fake_lengths.clip(upper=500), real_lengths.clip(upper=500)]
bp = axes[1].boxplot(data_to_plot, labels=['Fake (CG)', 'Real (OR)'],
                     patch_artist=True, medianprops=dict(color='white', linewidth=2))
bp['boxes'][0].set_facecolor('#E05252')
bp['boxes'][1].set_facecolor('#2ecc71')
axes[1].set_ylabel('Review Length (words, capped at 500)', fontsize=11)
axes[1].set_title('Review Length Boxplot\nFake vs. Real', fontsize=14, fontweight='bold')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Fake — mean: {fake_lengths.mean():.1f} words,  median: {fake_lengths.median():.0f} words")
print(f"Real — mean: {real_lengths.mean():.1f} words,  median: {real_lengths.median():.0f} words")

# clean the data

In [7]:
# drop rows that have the text missing
reviews_df.dropna(subset=['text_'], inplace=True)

#convert float64 ratings to integers
reviews_df["rating"] = reviews_df["rating"].astype(int)

In [8]:
from sklearn.preprocessing import LabelEncoder

# reviews_df.dropna(how="any") use this if eliminate any row that contains an usable value. i.e.
# This is commented out because we want to keep rows that have feature other than text missing. To handle these values we will
# use imputer rather than just deleting entire row.

# convert the labels of Computer Generated and Original to 0 and 1.

label_encoder = LabelEncoder()
label_encoder.fit(["CG","OR"])
reviews_df['label'] = label_encoder.transform(reviews_df['label'])

reviews_df.head()



,category,rating,label,text_
0,Home_and_Kitchen_5,5,0,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5,0,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5,0,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1,0,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5,0,Very nice set. Good quality. We have had the s...


# Split data into training and test data

In [9]:
# creates a split object, that represents 1 split, with 20% of the data being used for testing and 80 for training.
# random_state could be any number, it just ensures that the split is reproducible,
# meaning that if you run the code multiple times, you will get the same split each time.
reviews_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# reviews_split makes sure there is similar or same percentage of training and test sets.
# i.e. training set wont have 90% CG labels while testing set only have 10% CG labels.
# train_index and test_index are two NumPy arrays of row indices
# we get the data frame for both sets by passing it into the locate function of our original dataframe
for train_index, test_index in reviews_split.split(reviews_df, reviews_df["label"]):
  training_set = reviews_df.loc[train_index]
  testing_set = reviews_df.loc[test_index]

# x and y train are used for supervised learning. Labels shown to the model
x_train = training_set.drop(columns = ["label"])
y_train = training_set["label"]

# x and y test used for evaluated the trained model on supervised learning. Labels are not shown to the model.
x_test = testing_set.drop(columns = ['label'])
y_test = testing_set["label"]

display(x_train.info(),y_train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 32345 entries, 25582 to 33847
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  32345 non-null  object
 1   rating    32345 non-null  int64 
 2   text_     32345 non-null  object
dtypes: int64(1), object(2)
memory usage: 1010.8+ KB
<class 'pandas.core.series.Series'>
Index: 32345 entries, 25582 to 33847
Series name: label
Non-Null Count  Dtype
--------------  -----
32345 non-null  int64
dtypes: int64(1)
memory usage: 505.4 KB


None

None

In [24]:
y_test

,label
33059,1
19898,0
38225,1
29581,0
1079,0
...,...
35089,1
39139,0
39084,0
28164,0


# Step 4: Prepare the Data for Machine Learning algorithms

## Data preprocessing pipelines (transformations)

In [ ]:
#Use one-hot encodign to conver the category strings in the category column to numerical values that the model can underestand
#Use scaling to scale rating numerical
#Use TF-IDF vectorizer to convert the review text into numerical features that the model can understand
#convert ratings to integers.


# note: might use small LLM to embed values for the text feature instead of TF-IDF to see which one gives more accurate results.

In [10]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

rating_feature = ['rating']
category_feature = ['category']
text_feature = 'text_'

#imputer handles the transformation of features that are either null or nan.
rating_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())

])
# one-hot converts column category into multiple columns with names of all values in the data. e.g. is_car, is_phone, is_radio instead of Category. Value in said rows
# are either 0 or 1 representing yes or no.
# unknown = "ignore" means that it will set all unknown values of new instances to 0s across the board for all the columns.
category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value = "missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

text_pipeline = Pipeline([
  # not needed since we used dropna for all of invalid text values earlier
  # ("imputer", SimpleImputer(strategy="constant", fill_value="missing"))

  #tf-idf converts a string feature to a row of numbers. Each between 0-1 and represent each word.
  # i.e. Each new column name is each word, instead of a single column named text
  ('tf-idf', TfidfVectorizer(stop_words="english", max_features=5000))

])

preprocessing = ColumnTransformer([
    ("ratings", rating_pipeline, rating_feature),
    ("categories", category_pipeline, category_feature),
    ("text", text_pipeline, text_feature)
])

In [ ]:
# Manual implementaion of one-hot encoder. Column Transformer actually does this automatically. Can delete this block, but i kept it
# for reference

# one-hot encoding for category feature
from sklearn.preprocessing import OneHotEncoder
# set sparse to False to get a dense array(numpy array object) instead of a sparse matrix
# handle_uknown will ignore any categories in the test set that were not seen in the training set and will not raise an error when evaluating categories that have not been seen.
category_encoder = OneHotEncoder(sparse_output=False, handle_unknown = 'ignore')

# creates sparse matrix of one-hot encoded values for the category column
category_1hot_columns = category_encoder.fit_transform(reviews_df[['category']])
category_1hot_columns


# creates a new dataframe with the one-hot encoded values and column names base on the original category names
category_1hot_df = pd.DataFrame(category_1hot_columns, columns=category_encoder.get_feature_names_out())

# add the new columns to the original dataframe
reviews_df = pd.concat([reviews_df, category_1hot_df], axis =1)
reviews_df.head()
# delete the original "category"  column since we now have the one-hot encoded columns
reviews_df.drop(columns = ['category'])




In [11]:
reviews_df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5,0,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5,0,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5,0,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1,0,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5,0,Very nice set. Good quality. We have had the s...


# Step 5: Select a Model and Train it

## Baseline Model
we use the dummy model, because it is the minimum level intelligence that we need to beat to be considered useful



In [12]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate


baseline_pipeline = Pipeline([
  ('prep', preprocessing),
  ('dummymodel', DummyClassifier(strategy="most_frequent"))
])

scoring = cross_validate(
  baseline_pipeline,
  x_train, y_train, scoring= ["accuracy", "precision", "recall", "average_precision"], cv = 5
)

scoring


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'fit_time': array([1.22086668, 1.25745368, 1.916394  , 1.28772569, 1.61990547]),
 'score_time': array([0.61673808, 0.62063146, 1.05940962, 0.60293221, 1.48586512]),
 'test_accuracy': array([0.49992271, 0.49992271, 0.49992271, 0.49992271, 0.49992271]),
 'test_precision': array([0.        , 0.        , 0.        , 0.49992271, 0.49992271]),
 'test_recall': array([0., 0., 0., 1., 1.]),
 'test_average_precision': array([0.50007729, 0.50007729, 0.50007729, 0.49992271, 0.49992271])}

## Inital Support Vector Machine

In [ ]:
from sklearn.svm import SVC, LinearSVC

# We use LinearSVC here for the initial test because it trains in seconds on large
# text datasets (O(n) complexity vs. O(n^2-3) for SVC with RBF kernel).
# GridSearchCV in Step 6 will search over both linear and rbf kernels to find the best one.

fake_review_classifier = make_pipeline(preprocessing, LinearSVC(max_iter=2000, random_state=42))
fake_review_classifier.fit(x_train, y_train)

initial_acc = accuracy_score(y_test, fake_review_classifier.predict(x_test))
print(f"Initial LinearSVC accuracy (before tuning): {initial_acc:.4f}  ({initial_acc:.2%})")

## Cross validate to check scores of the initial SVM

In [ ]:
# Cross-validate the initial LinearSVC pipeline to get robust score estimates.
# NOTE: Previously this cell accidentally used baseline_pipeline (the dummy classifier)
# instead of fake_review_classifier — that bug is now fixed.

SVM_scoring = cross_validate(
    fake_review_classifier,
    x_train, y_train,
    scoring=["accuracy", "precision", "recall", "average_precision"],
    cv=5
)

print("Cross-validation results (Initial LinearSVC):")
print("-" * 48)
for metric in ["test_accuracy", "test_precision", "test_recall", "test_average_precision"]:
    mean  = SVM_scoring[metric].mean()
    label = metric.replace("test_", "").replace("_", " ").title()
    print(f"  {label:<25}: {mean:.4f}  ({mean:.2%})")

print(f"\nMean average precision: {SVM_scoring['test_average_precision'].mean():.4f}")

# Step 6: Fine-tune the Model

In [ ]:
# ── Step 6: Fine-tune the Model with GridSearchCV ──────────────────────────
#
# WHY WE SUBSAMPLE:
# SVC with RBF kernel has O(n²–n³) training complexity. Running GridSearchCV
# on all 32K training rows × 12 param combos × 5 folds = 60 fits would take
# hours. Instead we:
#   1. Sample a stratified 8K-row subset to quickly find the best params.
#   2. Retrain the winner on the FULL training set for maximum accuracy.

from sklearn.model_selection import StratifiedShuffleSplit

# ── 1. Draw a stratified 8K-row search subset ─────────────────────────────
search_split = StratifiedShuffleSplit(n_splits=1, test_size=0.75, random_state=42)
for search_idx, _ in search_split.split(x_train, y_train):
    x_search = x_train.iloc[search_idx]
    y_search = y_train.iloc[search_idx]

print(f"Hyperparameter search subset: {len(x_search):,} rows  "
      f"(label balance: {y_search.mean():.2%} real)")

# ── 2. Grid search on the subset ──────────────────────────────────────────
svc_pipeline = make_pipeline(preprocessing, SVC())

param_grid = {
    "svc__C":      [0.1, 1, 10],
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma":  ["scale", "auto"],
}

grid_search = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="average_precision",
    n_jobs=-1,
    verbose=2,
)

print("\nRunning GridSearchCV on subset (this should take ~1–3 minutes)…")
grid_search.fit(x_search, y_search)

print("\nBest parameters found:", grid_search.best_params_)
print(f"Best CV score (on subset): {grid_search.best_score_:.4f}")

# ── 3. Retrain best config on the FULL training set ───────────────────────
best = {k.replace("svc__", ""): v for k, v in grid_search.best_params_.items()}
final_svc_pipeline = make_pipeline(preprocessing, SVC(**best))

print(f"\nRetraining SVC({best}) on full {len(x_train):,}-row training set…")
final_svc_pipeline.fit(x_train, y_train)

# Patch grid_search so downstream cells that call grid_search.predict() still work
grid_search.best_estimator_ = final_svc_pipeline

full_acc = accuracy_score(y_test, final_svc_pipeline.predict(x_test))
print(f"Final SVM accuracy (full training set): {full_acc:.4f}  ({full_acc:.2%})")

In [26]:
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

Best parameters: {'svc__C': 10, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
Best cross-validation score: 0.9358440007115021


# PCA Pipeline to visualize model

In [16]:

data_visualization_pipeline = Pipeline([
    ('preprocessing', preprocessing), # This is the object that produced the table above
    ('pca', PCA(n_components=2))
])

# Perform PCA
x_train_pca = data_visualization_pipeline.fit_transform(x_train)

# Visualize weight of features in a dimensional space of 2.

In [17]:
# Pull out the PCA obeject.

pca_step = data_visualization_pipeline.named_steps['pca']

# GET NEW COLUMN NAMES
# Get new features that were made with preprocessing column transformer.
feature_names = data_visualization_pipeline.named_steps['preprocessing'].get_feature_names_out()


# create the loadings DataFrame : tells you exactly how much each of the original features (like Rating or the word "Excellent") contributed to the two new PCA axes.
# PC1 and PC2 because we chose n_components = 2, dimensional space of 2. PC1 = x axis and PC2 = y-axis. Each data point represents an instance i.e. (PC1,PC2)
# a high number in PC1 let, for example rating lets say has .90, means that rating has a strong influence on PC1(x-axis) to move in the right direction on the graph.
loadings_df = pd.DataFrame(
    pca_step.components_,
    columns=feature_names,
    index=['PC1', 'PC2']
)
# 4. Instead of just display(loadings_df), show the TOP 10 features because our text column is broken down into many many columns we cant just print it all out.
print("--- Top 10 Features for PC1 (X-axis) ---")
print(loadings_df.T['PC1'].sort_values(ascending=False).head(10))

print("\n--- Top 10 Features for PC2 (Y-axis) ---")
print(loadings_df.T['PC2'].sort_values(ascending=False).head(10))

display(loadings_df)

--- Top 10 Features for PC1 (X-axis) ---
ratings__rating                                      0.999510
categories__category_Kindle_Store_5                  0.013159
categories__category_Toys_and_Games_5                0.008024
text__love                                           0.007458
text__great                                          0.007225
categories__category_Tools_and_Home_Improvement_5    0.005247
categories__category_Sports_and_Outdoors_5           0.004956
text__loves                                          0.004415
text__perfect                                        0.003103
text__loved                                          0.002489
Name: PC1, dtype: float64

--- Top 10 Features for PC2 (Y-axis) ---
categories__category_Kindle_Store_5    0.810825
categories__category_Books_5           0.277834
text__book                             0.175536
text__read                             0.113232
text__story                            0.108616
text__characters               

,ratings__rating,categories__category_Books_5,categories__category_Clothing_Shoes_and_Jewelry_5,categories__category_Electronics_5,categories__category_Home_and_Kitchen_5,categories__category_Kindle_Store_5,categories__category_Movies_and_TV_5,categories__category_Pet_Supplies_5,categories__category_Sports_and_Outdoors_5,categories__category_Tools_and_Home_Improvement_5,...,text__yrs,text__zero,text__zip,text__zipper,text__zippers,text__zoe,text__zombie,text__zombies,text__zone,text__zoom
PC1,0.99951,-0.000118,-0.008203,-0.008915,-0.000639,0.013159,-0.009690,-0.003821,0.004956,0.005247,...,0.000027,-0.000306,-0.000022,-0.000116,-0.000027,-0.000009,-0.000009,-0.000008,0.000002,-0.000068
PC2,-0.01273,0.277834,-0.122689,-0.148459,-0.165078,0.810825,-0.037284,-0.210884,-0.145078,-0.143465,...,-0.000243,-0.000410,-0.000234,-0.001475,-0.000276,0.000429,0.000622,0.000610,0.000057,-0.000565


# Plot 2D representation of Model

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(x_train_pca[:,0], x_train_pca[:,1], c=y_train, cmap="viridis", edgecolor='k', s=150)
plt.xlabel("Principle Component 1")
plt.ylabel("Principle Component 2")
plt.title("Scaled PCA Projection of Fake Review Detection (SVC)")
plt.colorbar(scatter)
plt.show()

Permuatation Importance to evaluate feature importance

In [20]:
results = permutation_importance(grid_search, x_test, y_test, n_repeats=3, random_state=42)
importances = results.importances_mean
print(importances)

[0.02711026 0.00870084 0.44057421]


In [ ]:
## Feature Importance Bar Chart (Permutation)

feature_names_pi = ['Star Rating', 'Product Category', 'Review Text (TF-IDF)']
sorted_idx   = np.argsort(results.importances_mean)
sorted_names = [feature_names_pi[i] for i in sorted_idx]
sorted_vals  = results.importances_mean[sorted_idx]
bar_colors   = ['#6C8EBF', '#B07FCC', '#0096B7']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(sorted_names, sorted_vals,
               color=[bar_colors[i] for i in sorted_idx],
               edgecolor='white', linewidth=0.8)
for bar in bars:
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', ha='left', fontsize=11, fontweight='bold')
ax.set_xlabel('Mean Accuracy Decrease (Permutation Importance)', fontsize=11)
ax.set_title('Feature Importance — Permutation Method', fontsize=14, fontweight='bold', pad=12)
ax.set_xlim(0, sorted_vals.max() * 1.18)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

Evaluate Final Model

In [22]:
# Hard predictions
y_test_pred = grid_search.predict(x_test)

# Probabilities (needed for ranking metrics)
#y_test_proba = grid_search.predict_proba(x_test)[:, 1]

display(y_test_pred)
#display(y_test_proba)

array([1, 0, 1, ..., 0, 0, 0])

Confusion Matrix

In [25]:
conf_matrix = confusion_matrix(y_test, y_test_pred)
class_report = classification_report(y_test, y_test_pred, target_names=['Fake','Real'])
print('Confusion Matrix:\n', conf_matrix)
print('Classification Report:\n', class_report)

Confusion Matrix:
 [[3528  516]
 [ 541 3502]]
Classification Report:
               precision    recall  f1-score   support

        Fake       0.87      0.87      0.87      4044
        Real       0.87      0.87      0.87      4043

    accuracy                           0.87      8087
   macro avg       0.87      0.87      0.87      8087
weighted avg       0.87      0.87      0.87      8087



In [ ]:
## Confusion Matrix Heatmap

class_names = ['Fake (CG)', 'Real (OR)']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='white', cbar=True, ax=ax,
            annot_kws={"size": 18, "weight": "bold"})
ax.set_xlabel('Predicted Labels', fontsize=13, labelpad=10)
ax.set_ylabel('True Labels', fontsize=13, labelpad=10)
ax.set_title('Confusion Matrix — Fake Review Classifier (SVM)', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(axis='both', labelsize=12)
plt.tight_layout()
plt.show()

## Classification Report Heatmap

class_report_dict = classification_report(
    y_test, y_test_pred, target_names=['Fake', 'Real'], output_dict=True
)
df_report  = pd.DataFrame(class_report_dict).transpose()
df_heatmap = df_report.iloc[:-1, :].drop(columns=['support'])

plt.figure(figsize=(10, 5))
sns.heatmap(df_heatmap, annot=True, fmt='.2f', cmap='Blues',
            cbar=False, linewidths=0.5, linecolor='white',
            vmin=0, vmax=1, annot_kws={"size": 14, "weight": "bold"})
plt.title('Classification Report', fontsize=14, fontweight='bold', pad=15)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0)
plt.tight_layout()
plt.show()

# Step 7: Analyzing the Finalized Model Against Evaluation Metrics and the Test Set

We compare the tuned SVM against the baseline and visualize the performance improvement.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_pred = np.zeros(len(y_test), dtype=int)  # dummy: always predicts 0

metrics_summary = {
    'Model':     ['Baseline (Dummy)', 'Tuned SVM (RBF, C=10)'],
    'Accuracy':  [f'{accuracy_score(y_test, baseline_pred):.2%}',  f'{accuracy_score(y_test, y_test_pred):.2%}'],
    'Precision': [f'{precision_score(y_test, baseline_pred, zero_division=0):.2%}', f'{precision_score(y_test, y_test_pred):.2%}'],
    'Recall':    [f'{recall_score(y_test, baseline_pred, zero_division=0):.2%}',    f'{recall_score(y_test, y_test_pred):.2%}'],
    'F1-Score':  [f'{f1_score(y_test, baseline_pred, zero_division=0):.2%}',        f'{f1_score(y_test, y_test_pred):.2%}'],
}

df_summary = pd.DataFrame(metrics_summary).set_index('Model')
print("=" * 55)
print("       MODEL PERFORMANCE COMPARISON")
print("=" * 55)
display(df_summary)

baseline_acc = accuracy_score(y_test, baseline_pred)
svm_acc      = accuracy_score(y_test, y_test_pred)
print(f"\nImprovement over baseline: +{(svm_acc - baseline_acc)*100:.1f} percentage points")
print(f"Best CV score (GridSearchCV): {grid_search.best_score_:.2%}")
print(f"Best hyperparameters: {grid_search.best_params_}")

# ── Side-by-side comparison bar chart ──
metrics_names   = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
baseline_scores = [accuracy_score(y_test, baseline_pred),
                   precision_score(y_test, baseline_pred, zero_division=0),
                   recall_score(y_test, baseline_pred, zero_division=0),
                   f1_score(y_test, baseline_pred, zero_division=0)]
svm_scores      = [accuracy_score(y_test, y_test_pred),
                   precision_score(y_test, y_test_pred),
                   recall_score(y_test, y_test_pred),
                   f1_score(y_test, y_test_pred)]

x, width = np.arange(len(metrics_names)), 0.35
fig, ax  = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline (Dummy)',       color='#B0C4DE', edgecolor='white')
bars2 = ax.bar(x + width/2, svm_scores,      width, label='Tuned SVM (RBF, C=10)', color='#0096B7', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2%}', ha='center', va='bottom', fontsize=9, color='#555')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2%}', ha='center', va='bottom', fontsize=9, fontweight='bold', color='#0F2B4C')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Baseline vs. Tuned SVM — Evaluation Metrics', fontsize=14, fontweight='bold', pad=12)
ax.set_xticks(x); ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_ylim(0, 1.12); ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Step 8: Save and Deploy the Model

We use `joblib` to serialize the trained pipeline (preprocessing + best SVM) to disk. Loading the full pipeline means raw review text can be classified instantly without re-running any preprocessing.

In [ ]:
import joblib, os

MODEL_PATH = 'fake_review_svm_model.pkl'
joblib.dump(grid_search.best_estimator_, MODEL_PATH)

size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"Model saved  →  {MODEL_PATH}  ({size_mb:.2f} MB)")
print(f"Type: {type(grid_search.best_estimator_).__name__}")
print(f"Best params: {grid_search.best_params_}")

# ── Load and run a sample prediction ──────────────────────────────────────
loaded_model = joblib.load(MODEL_PATH)
print(f"\nModel loaded successfully from: {MODEL_PATH}\n")

label_map = {0: 'Fake (CG)', 1: 'Real (OR)'}
examples = [
    ('This product is absolutely amazing! Best purchase ever. Five stars!', 5, 'Home_and_Kitchen_5'),
    ('Works as described but the handle gets warm after 10 minutes. Decent for the price.', 4, 'Home_and_Kitchen_5'),
]

print(f"{'Review':<55} {'Prediction'}")
print("-" * 70)
for text, rating, cat in examples:
    row  = pd.DataFrame([{'text_': text, 'rating': rating, 'category': cat}])
    pred = label_map[loaded_model.predict(row)[0]]
    print(f"{text[:52]:<55} {pred}")